This notebook reads IND.csv, USA.csv (CSV files) and AUS.xlsx (Excel file) from the local source directory and loads them into the SQL Server raw.Member_Profile_Raw table.

**Usage:** Run each cell in order.

Changes vs. the Excel-only version:
- IND.csv / USA.csv are read with `pd.read_csv(encoding="utf-8-sig", dtype=object)` — `utf-8-sig` strips a UTF-8 BOM if Excel wrote one, and is harmless if there is none.
- Column names are stripped of surrounding whitespace so JSON keys match what the SQL `JSON_VALUE(...)` paths expect (e.g. `$.ID`, `$.Name`, `$.EnrollmentDate`).
- Values are stringified; all-null rows are skipped so empty trailing lines in a CSV do not land as empty records.

### Run each cell in order to execute the full pipeline.

In [ ]:
# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

import os
import json
import uuid
import sys
import logging
import datetime

# Source files are in the session's uploads directory
SOURCE_PATH = r"D:\\Incubyte Exam"

FILES = {
    "IND": "IND.csv",
    "USA": "USA.csv",
    "AUS": "AUS.xlsx",
}

SQL_SERVER = r"DESKTOP-ANIL"
SQL_DATABASE = "SkyPoints_DataEngineering"
SQL_DRIVER = "{ODBC Driver 18 for SQL Server}"

CONNECTION_STRING = (
    f"DRIVER={SQL_DRIVER};"
    f"SERVER={SQL_SERVER};"
    f"DATABASE={SQL_DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# ---------------------------------------------------------------------------
# LOGGING SETUP
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()],
)

log = logging.getLogger(__name__)

print("Configuration loaded successfully.")
print(f"Source path: {SOURCE_PATH}")
print(f"Files: {FILES}")
print(f"SQL Server: {SQL_SERVER} / {SQL_DATABASE}")

In [ ]:
# ---------------------------------------------------------------------------
# convert_value: Convert pandas values to JSON-safe Python types
# ---------------------------------------------------------------------------

def convert_value(value):
    """Convert a pandas value to a JSON-safe Python type."""
    if value is None:
        return None
    try:
        import pandas as pd
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, float):
        # CSV keeps everything text, but if something still parsed as float
        # (e.g. an ID column), stringify without scientific notation.
        if value == int(value):
            return str(int(value))
        return str(value)
    return value

print("convert_value function defined.")

In [ ]:
# ---------------------------------------------------------------------------
# MAIN ETL PIPELINE
# ---------------------------------------------------------------------------

FORCE_RELOAD = True   # temporary override for testing


def main():
    # Import pandas inside main to avoid import-time dependency issues
    import pandas as pd
    import pyodbc

    log.info("=" * 60)
    log.info("INCUBYTE - EXCEL/CSV TO SQL SERVER DATA LOAD PIPELINE")
    log.info("=" * 60)

    # --------------------------------------------------------------
    # 1. Validate source files exist
    # --------------------------------------------------------------
    for country, filename in FILES.items():
        file_path = os.path.join(SOURCE_PATH, filename)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Source file not found: {file_path}")
    log.info("All source files present.")

    # --------------------------------------------------------------
    # 2. Connect to SQL Server
    # --------------------------------------------------------------
    connection = pyodbc.connect(CONNECTION_STRING)
    cursor = connection.cursor()
    log.info(f"Connected to SQL Server: {SQL_SERVER} / {SQL_DATABASE}")

    # --------------------------------------------------------------
    # 3. Prevent duplicate loading (unless --force)
    # --------------------------------------------------------------
    cursor.execute("SELECT COUNT(*) FROM raw.Member_Profile_Raw")
    existing_count = cursor.fetchone()[0]

    if existing_count > 0:
        if FORCE_RELOAD:
            log.info(
                f"--force supplied: truncating raw.Member_Profile_Raw "
                f"({existing_count} existing records)."
            )
            cursor.execute("DELETE FROM raw.Member_Profile_Raw")
            connection.commit()
            log.info("Raw table cleared. Proceeding to reload.")
        else:
            raise Exception(
                f"RAW table already contains {existing_count} records. "
                "Loading stopped to prevent duplicates. "
                "Re-run with --force to clear and reload first."
            )
    else:
        log.info("RAW table empty — proceeding with load.")

    # --------------------------------------------------------------
    # 4. Process each country file
    # --------------------------------------------------------------
    total_loaded = 0
    BATCH_ID = str(uuid.uuid4())

    for country, filename in FILES.items():
        file_path = os.path.join(SOURCE_PATH, filename)
        log.info(f"Processing: {filename} (country={country})")

        # Choose reader based on file extension
        if filename.lower().endswith(".csv"):
            # utf-8-sig strips a leading BOM if Excel wrote one; harmless if absent.
            df = pd.read_csv(file_path, dtype=object, encoding="utf-8-sig")
            # Strip whitespace from column names so JSON keys match what the
            # SQL JSON_VALUE paths expect (e.g. $.ID, $.Name, $.EnrollmentDate).
            df.columns = [str(c).strip() for c in df.columns]
        else:
            df = pd.read_excel(file_path, dtype=object)

        # Strip whitespace from every string cell — CSV headers/values often
        # carry trailing spaces that would break JSON key matching downstream.
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].apply(
                    lambda v: v.strip() if isinstance(v, str) else v
                )

        # Drop fully-empty rows (e.g. trailing blank lines in CSV) so they
        # do not become empty records in the raw table.
        df = df.dropna(how="all")
        log.info(f"  Records found in {filename}: {len(df)}")

        insert_rows = []
        for index, row in df.iterrows():
            original_record = {
                str(column): convert_value(value)
                for column, value in row.items()
            }
            raw_json = json.dumps(
                original_record,
                ensure_ascii=False,
                default=str,
            )
            insert_rows.append((
                country,
                filename,
                index + 2,  # row number (1-indexed header + 0-base)
                raw_json,
                None,       # File_Date (will be set from source if present)
                BATCH_ID,
            ))

        if insert_rows:
            cursor.executemany("""
                INSERT INTO raw.Member_Profile_Raw
                (
                    Source_Country,
                    Source_File,
                    Source_Row_Number,
                    Raw_Record,
                    File_Date,
                    Batch_ID
                )
                VALUES (?, ?, ?, ?, ?, ?)
            """, insert_rows)

        total_loaded += len(insert_rows)
        log.info(f"  Records prepared & inserted: {len(insert_rows)}")

    # --------------------------------------------------------------
    # 5. Commit and close
    # --------------------------------------------------------------
    connection.commit()
    log.info("=" * 60)
    log.info("DATA LOADING COMPLETED SUCCESSFULLY")
    log.info("=" * 60)
    log.info(f"Total records loaded: {total_loaded}")
    log.info(f"Batch ID: {BATCH_ID}")

    cursor.close()
    connection.close()
    log.info("SQL Server connection closed.")


if __name__ == "__main__":
    main()